# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulmoiz-25/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My Lane as an ML Task (Type)

My selected lane is **Refresh / Content Opportunity Scoring**. The goal is to help content editors identify which existing pages should be reviewed first to improve their search performance and overall content quality.

I frame this as a **ranking/scoring** machine learning task. Instead of making a simple yes-or-no decision, the model assigns each content page a priority score based on its likelihood of benefiting from a refresh. Pages with higher scores are ranked at the top, allowing editors to focus on the most valuable opportunities first.

A ranking approach is more practical than binary classification because content teams usually have limited time and resources. Rather than reviewing every page, they can start with the highest-priority pages that are expected to have the greatest impact.

The model acts as a **decision-support tool**. It provides recommendations, while the final decision to update, merge, expand, or leave a page unchanged remains with the content editor.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

lane = "Refresh / Content Opportunity Scoring"

task_information = {
    "Lane": lane,
    "ML Task Type": "Ranking / Scoring",
    "Unit of Analysis": "One content page",
    "Primary User": "Content Editor / SEO Reviewer",
    "Model Output": "Prioritized list of pages for refresh",
    "Business Goal": "Help editors focus on pages with the highest improvement potential"
}

summary = pd.DataFrame(
    task_information.items(),
    columns=["Attribute", "Description"]
)

display(summary)

,Attribute,Description
0,Lane,Refresh / Content Opportunity Scoring
1,ML Task Type,Ranking / Scoring
2,Unit of Analysis,One content page
3,Primary User,Content Editor / SEO Reviewer
4,Model Output,Prioritized list of pages for refresh
5,Business Goal,Help editors focus on pages with the highest i...


## 2. Target or Proxy

The goal of this project is to identify content pages that are most likely to need a refresh in the future. Ideally, the model would predict whether a page will experience a noticeable decline in search performance during a future time period.

The target could be defined as **future_decline**, where:

- **1** = The page experiences a significant decline in future search performance.
- **0** = The page maintains or improves its performance.

Since the starter dataset does not contain future outcomes, I use **trend_direction** as a temporary **proxy target**. Pages with a value of **"down"** are assigned **1**, while pages with any other trend are assigned **0**.

This proxy is suitable for demonstrating the ML workflow, but it is not an ideal production target because it is derived from historical performance. Using the same information as both a feature and a target could introduce **target leakage**, allowing the model to learn an existing pattern instead of predicting future behaviour.

In a real production system, the target would be created from a future observation window so that the model learns to predict unseen outcomes rather than reproduce historical labels. This makes the predictions more reliable and useful for decision-making.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

# Locate the dataset
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path:
    df = pd.read_csv(data_path)
    data_source = str(data_path)
else:
    github_url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(github_url)
    data_source = github_url

print(f"Dataset Source : {data_source}")
print(f"Dataset Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")

# Create a temporary proxy target
df["future_decline_proxy"] = (
    df["trend_direction"]
      .astype(str)
      .str.lower()
      .eq("down")
      .astype(int)
)

print("\nProxy Target Distribution")
display(
    df["future_decline_proxy"]
      .value_counts()
      .rename_axis("Class")
      .reset_index(name="Count")
)

print("\nSample Records")
display(
    df[
        ["content_id", "trend_direction", "future_decline_proxy"]
    ].head(10)
)


Dataset Source : https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv
Dataset Shape  : 30,000 rows × 44 columns

Proxy Target Distribution


,Class,Count
0,1,16262
1,0,13738



Sample Records


,content_id,trend_direction,future_decline_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 3. Success Metric

The primary evaluation metric for this lane is **Precision@50**.

The model generates a ranked list of content pages, so the highest-ranked recommendations should provide the greatest value to content editors. Since editors usually review only a limited number of pages during each content review cycle, the quality of the top recommendations is more important than evaluating every page equally.

**Precision@50** measures the proportion of relevant pages within the first 50 recommendations. A higher Precision@50 indicates that the model is successfully placing the best refresh opportunities at the top of the ranking, allowing editors to spend their time on pages with the highest expected impact.

For this project, a **Precision@50 of 0.60 or higher** represents a strong starting point. This means that at least **30 of the top 50 recommended pages** are relevant according to the target or proxy label. The exact threshold may change as more historical data becomes available and business objectives evolve.

This metric aligns with the business goal of improving editorial efficiency by prioritizing the pages that are most likely to benefit from a content refresh rather than treating every page with equal importance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Evaluation metric for the ranking model

TOP_K = 50
TARGET_PRECISION = 0.60

# Calculate the base rate of the proxy target
proxy_rate = df["future_decline_proxy"].mean()
minimum_relevant = int(TOP_K * TARGET_PRECISION)

# Create a clean summary table
metric_summary = pd.DataFrame({
    "Metric": [
        "Evaluation Metric",
        "Top K Recommendations",
        "Target Precision",
        "Minimum Relevant Pages",
        "Proxy Positive Rate"
    ],
    "Value": [
        "Precision@50",
        TOP_K,
        f"{TARGET_PRECISION:.2f}",
        f"{minimum_relevant} of {TOP_K}",
        f"{proxy_rate:.3f}"
    ]
})

print("Success Metric Summary\n")
display(metric_summary)

Success Metric Summary



,Metric,Value
0,Evaluation Metric,Precision@50
1,Top K Recommendations,50
2,Target Precision,0.60
3,Minimum Relevant Pages,30 of 50
4,Proxy Positive Rate,0.542


## 4. The Unit of Analysis, as a Real DataFrame

The unit of analysis for this project is **one pseudonymized content page**. Each row in the dataset represents a single content page associated with a specific client. The dataset contains content characteristics, search performance metrics, engagement information, and trend indicators that can help estimate whether a page should be prioritized for a content refresh.

For this analysis, I created a working dataframe by selecting pages that:

- Have at least one search impression during the last 90 days.
- Are at least 90 days old.
- Contain the features required for this ML task.

The selected dataframe includes key attributes such as content type, search impressions, clicks, sessions, click-through rate (CTR), average search position, engagement rate, and the temporary proxy target (**future_decline_proxy**).

Each row represents one unique content page, making it the appropriate unit of analysis for building a ranking model that prioritizes refresh opportunities.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the working dataframe for the selected lane

selected_columns = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "future_decline_proxy"
]

# Keep only columns that exist in the dataset
selected_columns = [col for col in selected_columns if col in df.columns]

# Create the working dataframe
lane_df = (
    df.loc[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90),
        selected_columns
    ]
    .drop_duplicates(subset="content_id")
    .reset_index(drop=True)
)

# Dataset summary
summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Unique Content Pages",
        "Unique Clients",
        "Missing Values"
    ],
    "Value": [
        f"{lane_df.shape[0]:,}",
        lane_df.shape[1],
        f"{lane_df['content_id'].nunique():,}",
        lane_df["client_id"].nunique(),
        lane_df.isna().sum().sum()
    ]
})

print("Working DataFrame Summary\n")
display(summary)

print("\nFirst Five Records\n")
display(lane_df.head())

print("\nFeature Data Types\n")
display(lane_df.dtypes.reset_index().rename(
    columns={"index": "Feature", 0: "Data Type"}
))


Working DataFrame Summary



,Metric,Value
0,Rows,"30,000"
1,Columns,12
2,Unique Content Pages,"30,000"
3,Unique Clients,32
4,Missing Values,2374



First Five Records



,content_id,client_id,content_type,main_intent,content_age_days,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,future_decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,187,3803,29,17,0.76,10.6,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,445,15320,7,9,0.05,20.3,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,141,12581,11,11,0.09,36.5,0.00,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,463,11751,58,78,0.49,6.2,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,263,19140,24,145,0.13,44.0,0.00,1



Feature Data Types



,Feature,Data Type
0,content_id,object
1,client_id,object
2,content_type,object
3,main_intent,object
4,content_age_days,int64
5,impressions_90d,int64
6,clicks_90d,int64
7,sessions_90d,int64
8,ctr,float64
9,avg_position,float64


## 5. Why ML Beats a Fixed Rule Here

A simple rule-based approach can be created using fixed thresholds, for example:

- Refresh a page if **CTR is below 2%**.
- Refresh a page if **traffic has dropped by more than 20%**.
- Refresh a page if the **average search position is greater than 15**.

Although these rules are easy to understand, they only consider one or two features at a time. In practice, content performance depends on the interaction of many factors such as impressions, clicks, sessions, CTR, average position, engagement rate, content age, and search trends. A page with a low CTR may still perform well because it targets competitive keywords, while another page with a higher CTR may still require attention due to declining traffic or engagement.

A machine learning model can evaluate these features together and learn patterns from historical data instead of relying on fixed thresholds. This allows the model to identify pages that are more likely to benefit from a content refresh, even when the relationship between features is complex.

The purpose of the model is to **support content editors**, not replace them. It produces a ranked list of pages that deserve attention first, helping editors use their time more efficiently and make consistent, data-informed decisions.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Example of a simple rule-based baseline

rule_based = (
    (df["ctr"] < 0.02) &
    (df["avg_position"] > 15)
)

rule_summary = pd.DataFrame({
    "Metric": [
        "Total Content Pages",
        "Pages Recommended by Rule",
        "Recommendation Rate"
    ],
    "Value": [
        len(df),
        int(rule_based.sum()),
        f"{rule_based.mean() * 100:.2f}%"
    ]
})

print("Rule-Based Baseline Summary\n")
display(rule_summary)

print("\nAverage CTR by Trend Direction\n")
display(
    df.groupby("trend_direction")["ctr"]
      .mean()
      .round(3)
      .reset_index()
)

comparison = pd.DataFrame({
    "Aspect": [
        "Decision Method",
        "Features Used",
        "Flexibility",
        "Business Value"
    ],
    "Rule-Based Approach": [
        "Fixed thresholds",
        "One or two features",
        "Low",
        "May miss important opportunities"
    ],
    "Machine Learning Approach": [
        "Learns from historical patterns",
        "Many interacting features",
        "High",
        "Prioritizes pages with the greatest refresh potential"
    ]
})

print("\nRule-Based vs Machine Learning\n")
display(comparison)

Rule-Based Baseline Summary



,Metric,Value
0,Total Content Pages,30000
1,Pages Recommended by Rule,5733
2,Recommendation Rate,19.11%



Average CTR by Trend Direction



,trend_direction,ctr
0,down,0.324
1,flat,1.377
2,new,1.297
3,stable,0.517
4,up,0.566



Rule-Based vs Machine Learning



,Aspect,Rule-Based Approach,Machine Learning Approach
0,Decision Method,Fixed thresholds,Learns from historical patterns
1,Features Used,One or two features,Many interacting features
2,Flexibility,Low,High
3,Business Value,May miss important opportunities,Prioritizes pages with the greatest refresh po...


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.